# Debugging & Validating Cleaned Gini Dataset
before merging into final panel

## Python File (make edits here, transfer to actual file later)

In [ ]:

import pandas as pd
import glob
from pathlib import Path

################# COMBINING YEARLY FILES INTO ONE FILE #####################

# Collecting list of filenames of gini index files 
CSVs = glob.glob('../../data_raw/gini/*Data.csv')

# List to hold dataframes to be concatenated
df_list = []

# Looping over files
for file in CSVs:
    df = pd.read_csv(file, header = 1)
    
    # NOTE: ADD DATA VALIDATION CHECKS
        # maybe check row count for each file to ensure standardized
    
    # Dropping empty columns created in read_csv
    df = df.dropna(axis=1, how='all')

    # Slicing year out of filename to make year variable
    filename = Path(file).name
    file_year = filename[7:11] 
    assert 2006 <= int(file_year) <= 2026
    df['year'] = int(file_year)

    # Saving each dataframe in list
    df_list.append(df)

# Concatenating all years, sorting panel by year 
panel = pd.concat(df_list, ignore_index=True)
panel = panel.sort_values(by=['year','Geographic Area Name'])


################# GEOGRAPHIC STANDARDIZATION & CLEANING #####################
# Obtaining separated county and state names
panel[['COUNTY_NAME', 'STATE_NAME']] = panel['Geographic Area Name'].str.split(', ', expand=True)

# Removing excluded states from dataset
excluded_geo_units = ['Puerto Rico', 'Connecticut', 'Alaska']
panel = panel[~panel['STATE_NAME'].isin(excluded_geo_units)]

# Geographic Standardization for geographic crosswalk merge
#  ['Bedford city', 'Do?a Ana County', 'La Salle Parish', 'Shannon County']

# Dropping Bedford city
    # NOTE: need to recalculate gini for combined Bedford city/county to fit geographic standardization
    # dropping bedford city here in first stage of analysis
    # Dropping bedford county across all datasets in final panel construction stage for easy reversal
panel = panel[~((panel['COUNTY_NAME'] == 'Bedford city') & (panel['STATE_NAME'] == 'Virginia'))]

# Correcting Doña Ana County string error from 2010 ACS file
panel.loc[
    (panel['COUNTY_NAME'] == 'Do?a Ana County') &
    (panel['STATE_NAME'] == 'New Mexico'), 'COUNTY_NAME'
] = 'Doña Ana County'

# Correcting LaSalle Parish, LA string error from 2010 and 2011 gini files
panel.loc[
    (panel['COUNTY_NAME'] == 'La Salle Parish') &
    (panel['STATE_NAME'] == 'Louisiana'), 'COUNTY_NAME'
] = 'LaSalle Parish'

# Standardizing Shannon County/Oglala Lakota County name to Oglala Lakota County
panel.loc[
    (panel['COUNTY_NAME'] == 'Shannon County') &
    (panel['STATE_NAME'] == 'South Dakota'), 'COUNTY_NAME'
] = 'Oglala Lakota County'


################# MERGING GEOGRAPHIC CROSSWALK #####################

# Merging in geographic crosswalk
fips = pd.read_csv('../../data_intermediate/county_geo_id.csv',
                   dtype={'full_fips': str,
                        'state_fips': str,
                        'county_fips': str})

gini = pd.merge(panel,
                 fips,
                 on=['COUNTY_NAME', 'STATE_NAME'],
                 how='left',
                 indicator=True
)

# Validating merge
assert gini.loc[gini['_merge'] == 'left_only'].empty # no unmatched counties
assert gini[gini.duplicated(keep=False, subset=['year', 'full_fips'])].empty # no county-year duplicates

################# SAVING DATA #####################

## Debugging FIPS Matching

* Found 4 unmatched counties:
    * dropped bedford city (will drop bedford county in final panel) for first stage analysis
        * (need to sort out how to get accurate gini for combined beford county & city for separated years)
    * corrected string errors in Dona Ana County and LaSalle Parish
    * standardized pre-rename Shannon/Oglala Lakota County to current name (Oglala Lakota County)

* No unmatched observations remain!


In [165]:
# FINDING UNMATCHED/PROBLEMATIC COUNTIES IN GINI DATA
unmatched = gini.loc[gini['_merge'] == 'left_only']
# Found: # unmatched = number duplicate observations

print(unmatched['STATE_NAME'].unique())
    # problematic states: ['Virginia', 'New Mexico', 'Louisiana', 'South Dakota']

print(unmatched['COUNTY_NAME'].unique())
    # problematic counties: ['Bedford city', 'Do?a Ana County', 'La Salle Parish', 'Shannon County']


<StringArray>
[]
Length: 0, dtype: str
<StringArray>
[]
Length: 0, dtype: str


#### Bedford City

In [166]:
# SEARCHING FOR THESE COUNTIES IN GEOGRAPHIC CROSSWALK TO STANDARDIZE THEM
# 1) BEDFORD CITY 
bedford = fips.loc[fips['COUNTY_NAME'].str.contains('bedford', case = False, na = False)]
print(bedford)
    # Beford City, Virgnia is standardized to 'Bedford County'

print(gini.loc[(gini['COUNTY_NAME']=='Bedford city') & (gini['STATE_NAME']=='Virginia')])
    # Bedford city only exists in gini 2010-2013

print(gini.loc[(gini['COUNTY_NAME']=='Bedford County') & (gini['STATE']=='VA')])
    # Beford County exists in Gini 2010-2024

# NOTE 
# RESULT: Need to decide how to harmonize pre-2013 Bedford city into Bedford County
# Decision: drop bedford city/county in first stage baseline analysis, add back in later
    # dropping bedford city in this file
    # dropping bedford county across all files in final panel cleaning script so it's easy to reverse

     STATE state_fips county_fips     COUNTY_NAME    STATE_NAME  geo_idx  \
2241    PA         42         009  Bedford County  Pennsylvania       44   
2422    TN         47         003  Bedford County     Tennessee       48   
2822    VA         51         019  Bedford County      Virginia       52   

     full_fips county_name_match  
2241     42009    bedford county  
2422     47003    bedford county  
2822     51019    bedford county  
Empty DataFrame
Columns: [Geography, Geographic Area Name, Estimate!!Gini Index, Margin of Error!!Gini Index, year, COUNTY_NAME, STATE_NAME, STATE, state_fips, county_fips, geo_idx, full_fips, county_name_match, _merge]
Index: []
            Geography      Geographic Area Name  Estimate!!Gini Index  \
164    0500000US51019  Bedford County, Virginia                0.4250   
3269   0500000US51019  Bedford County, Virginia                0.4149   
6374   0500000US51019  Bedford County, Virginia                0.4149   
9479   0500000US51019  Bedford Co

#### Doña Ana County

In [167]:
# SEARCHING FOR THESE COUNTIES IN GEOGRAPHIC CROSSWALK TO STANDARDIZE THEM
# (2) DO?A ANA COUNTY

dona_ana = fips.loc[fips['county_name_match'].str.contains('dona ana', case = False, na = False)]
print(dona_ana)
    # Standardized to 'Doña Ana County' in crosswalk

print(gini.loc[(gini['COUNTY_NAME'].str.contains('Ana County', na=False)) & (gini['STATE_NAME'] == 'New Mexico')])
    # only 2010 file has 'Do?a Ana County' error, the rest are correct (Doña Ana County)

# NOTE
    # Decision: correct 2010 county name directly -- it will then match the crosswalk exactly


     STATE state_fips county_fips      COUNTY_NAME  STATE_NAME  geo_idx  \
1795    NM         35         013  Doña Ana County  New Mexico       37   

     full_fips county_name_match  
1795     35013   dona ana county  
            Geography         Geographic Area Name  Estimate!!Gini Index  \
785    0500000US35013  Do?a Ana County, New Mexico                0.4610   
3915   0500000US35013  Doña Ana County, New Mexico                0.4653   
7020   0500000US35013  Doña Ana County, New Mexico                0.4674   
10125  0500000US35013  Doña Ana County, New Mexico                0.4755   
13230  0500000US35013  Doña Ana County, New Mexico                0.4813   
16335  0500000US35013  Doña Ana County, New Mexico                0.4820   
19440  0500000US35013  Doña Ana County, New Mexico                0.4801   
22545  0500000US35013  Doña Ana County, New Mexico                0.4924   
25650  0500000US35013  Doña Ana County, New Mexico                0.4929   
28755  0500000US350

#### La Salle Parish

In [168]:
# SEARCHING FOR THESE COUNTIES IN GEOGRAPHIC CROSSWALK TO STANDARDIZE THEM
# (3) La Salle Parish

# Finding La Salle in geographic crosswalk
print(fips.loc[fips['county_name_match'].str.contains('salle', na=False)])
    # Found: written 'LaSalle Parish' in crosswalk

# Finding all La Salle Parish observatiosn in gini
print(gini.loc[
    (gini['COUNTY_NAME'].str.contains('Salle', case = False, na=False)) &
    (gini['STATE_NAME'] == 'Louisiana')
])
    # FOUND: 2010, 2011 observations are 'La Salle Parish'
    # Every other year is 'LaSalle Parish'

# NOTE
    # Decision: correct 2010 and 2011 county names to 'LaSalle Parish' in gini cleaning file


     STATE state_fips county_fips      COUNTY_NAME STATE_NAME  geo_idx  \
637     IL         17         099   LaSalle County   Illinois       19   
1135    LA         22         059   LaSalle Parish  Louisiana       24   
2657    TX         48         283  La Salle County      Texas       49   

     full_fips county_name_match  
637      17099    lasalle county  
1135     22059    lasalle parish  
2657     48283   la salle county  
            Geography        Geographic Area Name  Estimate!!Gini Index  \
1525   0500000US22059  La Salle Parish, Louisiana                0.4490   
4630   0500000US22059  La Salle Parish, Louisiana                0.4391   
7739   0500000US22059   LaSalle Parish, Louisiana                0.4363   
10844  0500000US22059   LaSalle Parish, Louisiana                0.4431   
13949  0500000US22059   LaSalle Parish, Louisiana                0.4628   
17054  0500000US22059   LaSalle Parish, Louisiana                0.4784   
20159  0500000US22059   LaSalle Parish

#### Shannon County, SD

In [169]:
# SEARCHING FOR THESE COUNTIES IN GEOGRAPHIC CROSSWALK TO STANDARDIZE THEM
# (4) Shannon County, South Dakota

print(gini.loc[
    (gini['COUNTY_NAME'].isin(['Shannon County', 'Oglala Lakota County'])) &
    (gini['STATE_NAME'] == 'South Dakota')
])
    # Shannon County changes to Oglala Lakota County starting in 2015, no overlap

# NOTE
    # Rename Shannon County to Oglala Lakota County pre- crosswalk-merge


            Geography                Geographic Area Name  \
2550   0500000US46113        Shannon County, South Dakota   
5655   0500000US46113        Shannon County, South Dakota   
8760   0500000US46113        Shannon County, South Dakota   
11865  0500000US46113        Shannon County, South Dakota   
14970  0500000US46113        Shannon County, South Dakota   
17614  0500000US46102  Oglala Lakota County, South Dakota   
20719  0500000US46102  Oglala Lakota County, South Dakota   
23824  0500000US46102  Oglala Lakota County, South Dakota   
26929  0500000US46102  Oglala Lakota County, South Dakota   
30034  0500000US46102  Oglala Lakota County, South Dakota   
33139  0500000US46102  Oglala Lakota County, South Dakota   
36244  0500000US46102  Oglala Lakota County, South Dakota   
39349  0500000US46102  Oglala Lakota County, South Dakota   
42454  0500000US46102  Oglala Lakota County, South Dakota   
45559  0500000US46102  Oglala Lakota County, South Dakota   

       Estimate!!Gini I

In [170]:
duplicate_obs = gini[gini.duplicated(keep=False, subset=['year', 'full_fips'])]